# 02 — Silver: Transformação PySpark + Delta Lake

**Responsabilidade:** Ler a Bronze, aplicar EDA, filtros de qualidade, cast de tipos e padronização de nomes para snake_case. Escrever em Delta Lake particionado.

> Execute `00_config` antes deste notebook.

## Célula 1 — Carregar configurações

In [ ]:
%run "./00_config"

## Célula 2 — EDA: análise exploratória

In [ ]:
df_raw = spark.table(BRONZE_TABLE)
print(f"Total raw: {df_raw.count():,}")
print("\n=== Estatisticas Descritivas ===")
df_raw.select("passenger_count", "total_amount").describe().show()
print("\n=== Valores Nulos ===")
cols = ['VendorID','passenger_count','total_amount',
        'tpep_pickup_datetime','tpep_dropoff_datetime']
df_raw.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in cols]).show()
print("\n=== Registros que serao removidos ===")
removals = {
    'passenger_count <= 0' : df_raw.filter(F.col('passenger_count') <= 0).count(),
    'total_amount <= 0'    : df_raw.filter(F.col('total_amount') <= 0).count(),
    'VendorID nulo'        : df_raw.filter(F.col('VendorID').isNull()).count(),
    'fora 2023 jan-mai'    : df_raw.filter(
        (F.col('tpep_pickup_datetime') < '2023-01-01') |
        (F.col('tpep_pickup_datetime') >= '2023-06-01')
    ).count(),
}
for motivo, qtd in removals.items():
    print(f"  {motivo:30s}: {qtd:,}")

## Célula 3 — Transformação: cast + qualidade + snake_case + partição

In [ ]:
# STEP 1: Cast explícito — uniformidade de tipos entre arquivos mensais
df_selected = df_raw.select(
    F.col('VendorID').cast(IntegerType()),
    F.col('passenger_count').cast(IntegerType()),
    F.col('total_amount').cast(DoubleType()),
    F.col('tpep_pickup_datetime').cast(TimestampType()),
    F.col('tpep_dropoff_datetime').cast(TimestampType()),
)

# STEP 2: Filtros de qualidade
# passenger_count > 0  : remove corridas vazias/teste
# total_amount > 0     : remove estornos e erros de registro
# VendorID NOT NULL    : remove registros corrompidos
# Periodo jan-mai 2023 : remove datas fora do escopo
df_clean = df_selected.filter(
    (F.col('passenger_count') > 0)
    & (F.col('total_amount') > 0)
    & F.col('VendorID').isNotNull()
    & (F.col('tpep_pickup_datetime') >= '2023-01-01')
    & (F.col('tpep_pickup_datetime') <  '2023-06-01')
)

# STEP 3: Padronizacao snake_case
# Remove prefixo tpep_ (codigo interno NYC TLC sem valor semantico)
# VendorID -> vendor_id seguindo convencao snake_case
df_renamed = df_clean \
    .withColumnRenamed('VendorID',             'vendor_id') \
    .withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

# STEP 4: Colunas de particao para partition pruning
df_silver = (
    df_renamed
    .withColumn('year',  F.year('pickup_datetime').cast('string'))
    .withColumn('month', F.month('pickup_datetime').cast('string'))
)

raw_count    = df_raw.count()
silver_count = df_silver.count()
removed      = raw_count - silver_count
print(f"Raw          : {raw_count:,}")
print(f"Silver       : {silver_count:,}")
print(f"Removidos    : {removed:,} ({removed/raw_count*100:.2f}%)")
print(f"\nColunas: {df_silver.columns}")
df_silver.printSchema()
df_silver.show(5, truncate=False)

## Célula 4 — Escrita Delta Lake no S3

In [ ]:
print(f"Escrevendo Silver -> {SILVER_S3}")
(
    df_silver.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('year', 'month')
    .save(SILVER_S3)
)
print("Silver escrita com sucesso!")

## Célula 5 — Registrar Silver no Unity Catalog

In [ ]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver')
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {SILVER_TABLE}
    USING DELTA
    LOCATION '{SILVER_S3}'
''')
print(f"Tabela: {SILVER_TABLE}")
spark.sql(f"DESCRIBE DETAIL {SILVER_TABLE}").select(
    "name","format","location","numFiles","sizeInBytes"
).show(truncate=False)
spark.sql(f'''
    ALTER TABLE {SILVER_TABLE}
    SET TAGS ('layer'='silver','domain'='nyc_taxi','year'='2023')
''')
print("Tags de governanca aplicadas")

## Célula 6 — OPTIMIZE + ZORDER

In [ ]:
# Compacta small files e cria indice de co-localizacao por data
# Queries com filtro temporal ficam ate 10x mais rapidas
print("Executando OPTIMIZE + ZORDER...")
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (pickup_datetime)")
print("Concluido")
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").select(
    "version","timestamp","operation","operationMetrics"
).show(5, truncate=False)

## Célula 7 — Validação Silver

In [ ]:
spark.sql(f"DESCRIBE TABLE {SILVER_TABLE}").show(truncate=False)
spark.sql(f'''
    SELECT year, CAST(month AS INT) AS mes, COUNT(*) AS total
    FROM {SILVER_TABLE} GROUP BY year, month ORDER BY year, mes
''').show()
spark.sql(f'''
    SELECT COUNT(*) AS total, ROUND(AVG(passenger_count),2) AS avg_pass,
           ROUND(AVG(total_amount),2) AS avg_amount,
           MIN(pickup_datetime) AS min_dt, MAX(pickup_datetime) AS max_dt
    FROM {SILVER_TABLE}
''').show(truncate=False)